# 🛠️ Colab Tools Backend — GPU-accelerated tasks for the bot

This notebook runs a FastAPI server on Colab GPU that exposes endpoints for:
- **NSFW classification** (image → safe/unsafe)
- **AI avatar detection** (image → AI-generated or not)
- **Background removal** (image → image without background)
- **Screenshot** (URL → screenshot image)
- **Speech-to-text** (audio → transcript)
- **Image generation** (text → image)
- **Embeddings** (text → vector for RAG)

The bot calls these endpoints instead of running them locally on the VPS.

**Instructions:**
1. Runtime > Change runtime type > GPU (T4)
2. Execute all cells
3. Copy the ngrok URL to the bot's `COLAB_TOOLS_URL` env var

In [ ]:
# Cell 1: Install dependencies
!pip install fastapi uvicorn pyngrok transformers torch Pillow requests
!pip install rembg playwright  # background removal + screenshots
!playwright install chromium
!pip install openai-whisper  # speech-to-text (optional, ~1.5GB)

In [ ]:
# Cell 2: Configuration
NGROK_AUTHTOKEN = ''  # <-- PASTE YOUR NGROK AUTHTOKEN
BOT_WEBHOOK_URL = ''  # <-- Bot webhook URL (optional, for auto-URL update)
PORT = 8000

# Model choices
NSFW_MODEL = 'Falconsai/nsfw_image_detection'  # lightweight NSFW classifier
AI_DETECT_MODEL = 'Organika/sdxl-detector'  # AI image detector
WHISPER_MODEL = 'base'  # tiny, base, small, medium, large

print(f'Config: NSFW={NSFW_MODEL}, AI_detect={AI_DETECT_MODEL}, Whisper={WHISPER_MODEL}')

In [ ]:
# Cell 3: Load models on GPU
import torch, time
from transformers import pipeline as hf_pipeline
from PIL import Image
import requests as req
from io import BytesIO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🎮 Device: {device}')

print('⏳ Loading NSFW classifier...')
nsfw_pipe = hf_pipeline('image-classification', model=NSFW_MODEL, device=device)
print('✅ NSFW classifier ready')

print('⏳ Loading AI image detector...')
ai_detect_pipe = hf_pipeline('image-classification', model=AI_DETECT_MODEL, device=device)
print('✅ AI detector ready')

# Whisper (optional, large download)
try:
    import whisper
    print(f'⏳ Loading Whisper {WHISPER_MODEL}...')
    whisper_model = whisper.load_model(WHISPER_MODEL).to(device)
    print('✅ Whisper ready')
except Exception as e:
    print(f'⚠️ Whisper not loaded: {e}')
    whisper_model = None

# Rembg (background removal)
try:
    from rembg import remove as rembg_remove
    print('✅ Rembg ready')
except:
    print('⚠️ Rembg not available')
    rembg_remove = None

print('\n✅ All models loaded on GPU')

In [ ]:
# Cell 4: FastAPI server
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel
import uvicorn, base64, io, asyncio
from PIL import Image
import requests as req

app = FastAPI(title='Colab Tools Backend', version='1.0.0')

class ImageRequest(BaseModel):
    url: str

class TextRequest(BaseModel):
    text: str

def load_image(url: str) -> Image.Image:
    resp = req.get(url, timeout=30)
    return Image.open(BytesIO(resp.content)).convert('RGB')

@app.get('/health')
async def health():
    return {'status': 'ok', 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}

@app.post('/nsfw')
async def classify_nsfw(req: ImageRequest):
    """Classify image as NSFW or safe."""
    try:
        img = load_image(req.url)
        result = nsfw_pipe(img)
        # Format: [{label: 'nsfw', score: 0.99}, {label: 'normal', score: 0.01}]
        scores = {r['label']: r['score'] for r in result}
        nsfw_score = scores.get('nsfw', scores.get('NSFW', 0))
        return {'is_nsfw': nsfw_score > 0.5, 'nsfw_score': nsfw_score, 'scores': scores}
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

@app.post('/ai-detect')
async def detect_ai_image(req: ImageRequest):
    """Detect if an image is AI-generated."""
    try:
        img = load_image(req.url)
        result = ai_detect_pipe(img)
        scores = {r['label']: r['score'] for r in result}
        ai_score = scores.get('ai', scores.get('AI', scores.get('fake', 0)))
        return {'is_ai': ai_score > 0.5, 'ai_score': ai_score, 'scores': scores}
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

@app.post('/remove-bg')
async def remove_bg(req: ImageRequest):
    """Remove background from image. Returns base64 PNG."""
    if rembg_remove is None:
        return JSONResponse({'error': 'rembg not available'}, status_code=503)
    try:
        img = load_image(req.url)
        result = rembg_remove(img)
        buf = BytesIO()
        Image.open(BytesIO(result)).save(buf, format='PNG')
        b64 = base64.b64encode(buf.getvalue()).decode()
        return {'image_base64': b64, 'format': 'png'}
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

@app.post('/screenshot')
async def take_screenshot(req: ImageRequest):
    """Take a screenshot of a URL using Playwright."""
    try:
        from playwright.async_api import async_playwright
        async with async_playwright() as p:
            browser = await p.chromium.launch(args=['--no-sandbox'])
            page = await browser.new_page(viewport={'width': 1280, 'height': 720})
            await page.goto(req.url, timeout=30000, wait_until='domcontentloaded')
            screenshot = await page.screenshot(full_page=False)
            await browser.close()
        b64 = base64.b64encode(screenshot).decode()
        return {'image_base64': b64, 'format': 'png'}
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

@app.post('/transcribe')
async def transcribe_audio(req: ImageRequest):
    """Transcribe audio from URL using Whisper."""
    if whisper_model is None:
        return JSONResponse({'error': 'whisper not loaded'}, status_code=503)
    try:
        # Download audio
        resp = req.get(req.url, timeout=60)
        with open('/tmp/audio_file', 'wb') as f:
            f.write(resp.content)
        result = whisper_model.transcribe('/tmp/audio_file')
        return {'text': result['text'], 'language': result.get('language', 'unknown')}
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

@app.post('/embeddings')
async def get_embeddings(req: TextRequest):
    """Generate text embeddings using Ollama (if running) or sentence-transformers."""
    try:
        # Try Ollama first
        try:
            resp = req.post('http://localhost:11434/api/embeddings',
                          json={'model': 'nomic-embed-text', 'prompt': req.text},
                          timeout=10)
            if resp.status_code == 200:
                return {'embedding': resp.json()['embedding'], 'model': 'nomic-embed-text'}
        except:
            pass
        # Fallback: sentence-transformers
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
        emb = model.encode(req.text).tolist()
        return {'embedding': emb, 'model': 'all-MiniLM-L6-v2'}
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

print('✅ FastAPI app configured with endpoints:')
print('  GET  /health')
print('  POST /nsfw')
print('  POST /ai-detect')
print('  POST /remove-bg')
print('  POST /screenshot')
print('  POST /transcribe')
print('  POST /embeddings')

In [ ]:
# Cell 5: Start ngrok + server
from pyngrok import ngrok, conf
import nest_asyncio, threading

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN

ngrok.kill()
time.sleep(2)

tunnel = ngrok.connect(PORT, 'http')
TOOLS_URL = tunnel.public_url
print(f'🌐 Colab Tools URL: {TOOLS_URL}')
print(f'   → Set COLAB_TOOLS_URL={TOOLS_URL} in bot .env')

# Notify bot webhook
if BOT_WEBHOOK_URL:
    try:
        resp = req.post(BOT_WEBHOOK_URL, json={'url': TOOLS_URL, 'type': 'tools'})
        print(f'📡 Bot notified: {resp.status_code}')
    except Exception as e:
        print(f'⚠️ Webhook failed: {e}')

# Start server in background
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print(f'✅ Server running on port {PORT}')
print(f'   Test: {TOOLS_URL}/health')

In [ ]:
# Cell 6: Keep-alive loop (same as LLM notebook)
import time, requests
from datetime import datetime, timedelta

REGENERATE_HOURS = 24
start_time = datetime.now()
regenerate_at = start_time + timedelta(hours=REGENERATE_HOURS)

print(f'🕐 Tools server started at {start_time.strftime("%H:%M:%S")}')
print(f'🔄 Will regenerate at {regenerate_at.strftime("%H:%M:%S")}')
print(f'📊 Keep-alive loop (checks every 60s)...')

ok_count = 0
fail_count = 0

try:
    while True:
        now = datetime.now()
        
        # Health check
        try:
            resp = requests.get(f'{TOOLS_URL}/health', timeout=10)
            if resp.status_code == 200:
                ok_count += 1
                status = '✅'
            else:
                fail_count += 1
                status = f'⚠️ {resp.status_code}'
        except:
            fail_count += 1
            status = '❌'
        
        # Status every 5 min
        elapsed = now - start_time
        if int(elapsed.total_seconds()) % 300 == 0 and int(elapsed.total_seconds()) > 0:
            print(f'[{now.strftime("%H:%M:%S")}] uptime={int(elapsed.total_seconds()/60)}min ok={ok_count} fail={fail_count} {status}')
        
        # Regenerate ngrok every 24h
        if now >= regenerate_at:
            print('🔄 Regenerating ngrok tunnel...')
            ngrok.kill()
            time.sleep(3)
            new_tunnel = ngrok.connect(PORT, 'http')
            TOOLS_URL = new_tunnel.public_url
            print(f'   New URL: {TOOLS_URL}')
            if BOT_WEBHOOK_URL:
                try:
                    requests.post(BOT_WEBHOOK_URL, json={'url': TOOLS_URL, 'type': 'tools'})
                    print('   📡 Bot notified')
                except Exception as e:
                    print(f'   ⚠️ Webhook failed: {e}')
            regenerate_at = now + timedelta(hours=REGENERATE_HOURS)
        
        time.sleep(60)
except KeyboardInterrupt:
    print('\n⏹️ Stopped')
except Exception as e:
    print(f'\n❌ Error: {e}')
finally:
    print(f'Stats: ok={ok_count} fail={fail_count}')